# IMPLEMENT THE NEURAL NETWORK FOR THE LEARNING OF CA TOPOLOGY

first I should learn and see convoca, for everything needed

In [6]:
 #!pip install timm

import torch
import numpy as np
import os
from PIL import Image
import numpy as np
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import Dataset
from torch.utils.data import DataLoader
import torchvision.utils as vutils
import torchvision.transforms.functional as TF
from sklearn.metrics import confusion_matrix
import seaborn as sns
from sklearn.metrics import roc_curve, roc_auc_score
import torch.nn.functional as F

# visualization
import matplotlib.pyplot as plt
import pandas as pd

from sklearn.metrics import classification_report

In [ ]:
import re

class CAdataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform  # CORRETTO: adesso memorizza il transform passato
        self.pairs = []
        self.labels = []
        
        # this can be better
        self.class_to_idx = {
            "moore": 0,
            "vn" : 1
        }
        
        def natural_sort_key(s):
            return [int(text) if text.isdigit() else text.lower() for text in re.split(r'(\d+)', s)]

        # Scansiona ricorsivamente tutte le sottocartelle (evitiamo di chiamarla 'root' per non fare confusione)
        for current_root, dirs, files in os.walk(self.root_dir):
            
            # lista dei files validi
            valid_files = [
                f for f in files
                if f.lower().endswith(('.png', '.jpg', '.jpeg')) and not f.startswith("._")
            ]     
            
            # cartella contiene almeno due elementi
            label_name = None
            if len(valid_files) >= 2:
                for class_name in self.class_to_idx:
                    if class_name in current_root.lower():
                        label_name = class_name
                        break
            
            if label_name == None:
                # we go on because current dir doesn't represent the right label
                continue
            
            label_idx = self.class_to_idx[label_name]
            valid_files.sort(key=natural_sort_key)
            
            for i in range(len(valid_files) -1):
                frame_t = os.path.join(current_root, valid_files[i])
                frame_t1 = os.path.join(current_root, valid_files[i+1])
                
                # (F0,F1,label)
                self.pairs.append((frame_t, frame_t1, label_idx))

    def __len__(self):
        
        # it basically return the length of the array image_paths
        return len(self.pairs)

    
    # important, in this case we have to gain a 2 touple (Ft,Ft+1) AS A SAMPLE for training
    # the NN
    def __getitem__(self, idx):
        
        # estraiamo la tupla
        frame_t, frame_t1, label_idx = self.pairs[idx]
        
        # open and convert the impages
        frame_t = Image.open(frame_t).convert('L')
        frame_t1 = Image.open(frame_t1).convert('L')
        
        # questo parametro è IMPORTANTISSIMO!!
        # dipende da scelte fatte nel notebook model, 
        # per ora teniamo 
        dim_logica = (20, 20)  
        frame_t = frame_t.resize(dim_logica, resample=Image.Resampling.NEAREST)
        frame_t1 = frame_t1.resize(dim_logica, resample=Image.Resampling.NEAREST)
        
        # Applica le trasformazioni se presenti
        if self.transform is not None:
            tensor_t = self.transform(frame_t)
            tensor_t1 = self.transform(frame_t1)
        else:
            tensor_t = TF.to_tensor(frame_t)
            tensor_t1 = TF.to_tensor(frame_t1)

        touple = torch.cat([tensor_t, tensor_t1,] , dim = 0)
        
        label = torch.tensor(label_idx, dtype=torch.long)

        
        # ritorna (T1,T2,label)
        return touple, label

I DATI VANNO ULTERIORMENTE PULITI, DOBBIAMO SERVIRCI DI TORCH, UNA 1000X1000 E' TROPPO GRANDE

In [19]:
CA_data = CAdataset(root_dir="dataset")

print(f"numero di coppie totali: {len(CA_data)}")

x,y = CA_data[0]
print(f"shape dell'input {x.shape}")
print(f"shape dell'output {y.shape}")

numero di coppie totali: 43
shape dell'input torch.Size([2, 20, 20])
shape dell'output torch.Size([])
